# Riconoscimento rifiuti 🗑️
Prima di iniziare: **Runtime → Cambia tipo di runtime → GPU (T4)**. Poi premi ▶ su ogni cella, in ordine.

## 1. Scarica le foto (ci mette qualche minuto, non fermarlo)

In [ ]:
%pip install -q kagglehub
import kagglehub

percorso = kagglehub.dataset_download('rfabrik/trashpanda')
print('Foto scaricate in:', percorso)

## 2. Carica le foto
Ogni foto viene rimpicciolita a 128x128. Il tipo di rifiuto è il nome della cartella in cui sta la foto.

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image

foto = []       # le immagini
nomi = []       # il tipo di rifiuto di ogni immagine

for file in Path(percorso).rglob('*'):
    if file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        immagine = Image.open(file).convert('RGB').resize((128, 128))
        foto.append(np.array(immagine))
        nomi.append(file.parent.name)

foto = np.array(foto)
classi = sorted(set(nomi))          # i tipi di rifiuto, in ordine alfabetico
numeri = np.array([classi.index(n) for n in nomi])

print(len(foto), 'foto')
print('Tipi di rifiuto:', classi)

## 3. Dividi le foto
80% per studiare (train), 10% per controllare durante lo studio (val), 10% per l'esame finale (test).

In [ ]:
from sklearn.model_selection import train_test_split

foto_train, foto_resto, num_train, num_resto = train_test_split(foto, numeri, test_size=0.2, stratify=numeri, random_state=42)
foto_val, foto_test, num_val, num_test = train_test_split(foto_resto, num_resto, test_size=0.5, stratify=num_resto, random_state=42)

print(len(foto_train), 'per studiare,', len(foto_val), 'per controllare,', len(foto_test), 'per l\'esame')

## 4. Costruisci la rete
Ogni blocco guarda la foto con dei filtri e la rimpicciolisce. Più blocchi = la rete capisce cose più complesse.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def blocco(filtri):
    return [
        layers.Conv2D(filtri, 3, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2D(filtri, 3, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(),
        layers.Dropout(0.1),
    ]

rete = keras.Sequential(
    [keras.Input((128, 128, 3)), layers.Rescaling(1 / 255)]      # i colori da 0-255 a 0-1
    + blocco(32) + blocco(64) + blocco(128) + blocco(256) + blocco(256)
    + [
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(len(classi), activation='softmax'),         # una risposta per ogni tipo di rifiuto
    ]
)
rete.summary()

## 5. Addestra
La rete guarda le foto tante volte (epoche) e impara. Per aiutarla a non imparare a memoria, giro e ingrandisco le foto a caso.

⏱️ Circa 45 minuti. Si ferma da sola se smette di migliorare.

In [ ]:
import math

# modifiche casuali alle foto (solo durante lo studio)
modifiche = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
])

EPOCHE = 60
BATCH = 64
passi = EPOCHE * math.ceil(len(foto_train) / BATCH)

modello = keras.Sequential([keras.Input((128, 128, 3)), modifiche, rete])
modello.compile(
    optimizer=keras.optimizers.AdamW(keras.optimizers.schedules.CosineDecay(2e-3, passi), weight_decay=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

modello.fit(
    foto_train, num_train,
    validation_data=(foto_val, num_val),
    epochs=EPOCHE, batch_size=BATCH, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)],
)

## 6. L'esame finale
Foto che la rete non ha mai visto.

In [ ]:
previsioni = rete.predict(foto_test, verbose=0).argmax(axis=1)
giuste = (previsioni == num_test).mean()
print(f'Risposte giuste: {giuste:.1%}')

## 7. Salva per il computer o il Raspberry
Scarica due file: `modello_rifiuti.tflite` e `etichette.txt`.

In [ ]:
import tensorflow as tf
from google.colab import files

convertitore = tf.lite.TFLiteConverter.from_keras_model(rete)
convertitore.optimizations = [tf.lite.Optimize.DEFAULT]
open('modello_rifiuti.tflite', 'wb').write(convertitore.convert())
open('etichette.txt', 'w').write('\n'.join(classi))

files.download('modello_rifiuti.tflite')
files.download('etichette.txt')